**SHARK ATTACKS CLEANING PIPELINE**

In [1]:
import pandas as pd
import logging

**LOGGING CONFIGURATION**

In [11]:
logging.basicConfig(
    level=logging.INFO,
    format='%(levelname)s - %(message)s'
)

**LOAD DATA**

In [27]:
df = pd.read_csv(
    "../data/raw/attacks.csv",
    encoding="latin1"
)

In [28]:
logging.info(f"Dataset loaded with shape: {df.shape}")

INFO - Dataset loaded with shape: (25614, 22)


**CLEANING FUNCTIONS**

In [29]:
def clean_column_names(df: pd.DataFrame) -> pd.DataFrame:
    """
    Clean column names:
    - remove spaces
    - lowercase
    - replace spaces with underscore
    """

    df = df.copy()

    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
    )

    logging.info("Column names cleaned")

    return df

In [30]:
def remove_blank_rows(df: pd.DataFrame) -> pd.DataFrame:
    """
    Remove fully empty rows.
    """

    df = df.copy()

    before = len(df)

    df = df.dropna(how='all')

    after = len(df)

    logging.info(f"Removed {before - after} blank rows")

    return df
    

In [31]:
def clean_fatal_column(df: pd.DataFrame) -> pd.DataFrame:
    """
    Standardize fatal_(y/n) values.
    """

    df = df.copy()

    df['fatal_(y/n)'] = (
        df['fatal_(y/n)']
        .astype(str)
        .str.strip()
        .str.upper()
    )

    logging.info("Fatal column standardized")

    return df

In [32]:
def drop_metadata_columns(df: pd.DataFrame) -> pd.DataFrame:
    """
    Remove metadata columns.
    """

    df = df.copy()

    metadata_cols = [
        'pdf',
        'href',
        'href_formula',
        'case_number.1',
        'case_number.2'
    ]

    df = df.drop(columns=metadata_cols)

    logging.info("Metadata columns removed")

    return df

In [33]:
def remove_duplicates(df: pd.DataFrame) -> pd.DataFrame:
    """
    Remove duplicate rows.
    """

    df = df.copy()

    duplicates = df.duplicated().sum()

    df = df.drop_duplicates()

    logging.info(f"Removed {duplicates} duplicate rows")

    return df

In [34]:
def analyze_missing_values(df: pd.DataFrame) -> pd.DataFrame:
    """
    Print missing values percentages.
    """

    missing_percent = (
        df.isnull().mean() * 100
    ).sort_values(ascending=False)

    print("\nMissing Values Percentage:\n")
    print(missing_percent)

    logging.info("Missing values analyzed")

    return df

**VALIDATION FUNCTION**

In [35]:
def validate_dataset(df: pd.DataFrame) -> pd.DataFrame:
    """
    Validate cleaned dataset.
    """

    assert len(df) > 0, "Dataset is empty!"

    assert 'pdf' not in df.columns

    assert df.duplicated().sum() == 0, \
        "Duplicates still exist!"

    logging.info(
        f"Validation passed: {len(df)} rows, {df.shape[1]} columns"
    )

    print("\n✓ Dataset validated successfully")

    return df

**PIPELINE**

In [36]:
clean_df = (
    df
    .pipe(clean_column_names)
    .pipe(remove_blank_rows)
    .pipe(clean_fatal_column)
    .pipe(remove_duplicates)
    .pipe(drop_metadata_columns)
    .pipe(analyze_missing_values)
    .pipe(validate_dataset)
)

INFO - Column names cleaned
INFO - Removed 19518 blank rows
INFO - Fatal column standardized
INFO - Removed 0 duplicate rows
INFO - Metadata columns removed
INFO - Missing values analyzed
INFO - Validation passed: 6096 rows, 17 columns



Missing Values Percentage:

time                      53.280840
species                   49.146982
age                       44.652231
sex                        9.498031
activity                   8.809055
location                   8.415354
area                       6.791339
name                       3.412073
country                    0.787402
fatal_(y/n)                0.524934
injury                     0.492126
investigator_or_source     0.311680
type                       0.098425
year                       0.065617
date                       0.032808
original_order             0.032808
case_number                0.016404
dtype: float64

✓ Dataset validated successfully


**FINAL OUTPUT**

In [37]:
print("\nFinal Shape:")
print(clean_df.shape)


Final Shape:
(6096, 17)


In [38]:
print("\nCleaned Dataset Preview:")
print(clean_df.head())


Cleaned Dataset Preview:
    case_number                  date    year        type         country  \
0    2017.06.11            2017-06-11  2017.0  Unprovoked       AUSTRALIA   
1  2017.06.10.b            2017-06-10  2017.0  Unprovoked       AUSTRALIA   
2  2017.06.10.a            2017-06-10  2017.0  Unprovoked             USA   
3  2017.06.07.R  Reported 07-Jun-2017  2017.0  Unprovoked  UNITED KINGDOM   
4    2017.06.04            2017-06-04  2017.0  Unprovoked             USA   

                area                                         location  \
0  Western Australia                         Point Casuarina, Bunbury   
1           Victoria                    Flinders, Mornington Penisula   
2            Florida                      Ponce Inlet, Volusia County   
3        South Devon                                    Bantham Beach   
4            Florida  Middle Sambo Reef off Boca Chica, Monroe County   

        activity            name sex  age  \
0  Body boarding       Paul

**EXPORT CLEAN DATASET**

In [39]:
clean_df.to_csv(
    "../data/processed/cleaned_attacks.csv",
    index=False
)

logging.info("Cleaned dataset exported successfully")

print("\n✓ Cleaned dataset saved successfully")

INFO - Cleaned dataset exported successfully



✓ Cleaned dataset saved successfully
